In [8]:
import pandas as pd
import numpy as np
import kagglehub
import os

path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")
print(f"Downloaded dataset path: {path}")

# The dataset file has been identified as 'cardekho_dataset.csv'
file_path = os.path.join(path, "cardekho_dataset.csv")
df = pd.read_csv(file_path)
df.info()
df.isnull().sum() * 100 / len(df)
df = df.dropna(subset=['selling_price'])
for col in ['mileage', 'engine', 'max_power']:
    df[col] = df[col].astype(str).str.extract('(\d+\.?\d*)')
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col].fillna(df[col].median(), inplace=True)


df = df[(df['selling_price'] != 999999999) & (df['selling_price'] >= 10000)]


df = df.drop_duplicates()


print("Shape after cleaning:", df.shape)

<>:16: SyntaxWarning: invalid escape sequence '\d'
<>:16: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_2931/2424253271.py:16: SyntaxWarning: invalid escape sequence '\d'
  df[col] = df[col].astype(str).str.extract('(\d+\.?\d*)')


Using Colab cache for faster access to the 'cardekho-used-car-data' dataset.
Downloaded dataset path: /kaggle/input/cardekho-used-car-data
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 no

/tmp/ipykernel_2931/2424253271.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_2931/2424253271.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

In [10]:
df['transmission_type'] = df['transmission_type'].map({'Manual': 0, 'Automatic': 1})

In [12]:
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

# Print columns
print(df.columns.tolist())

['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Define X and y
X = df.drop('selling_price', axis=1)
y = df['selling_price']


X = X.select_dtypes(include=[np.number])



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


baseline_pred = [y_train.mean()] * len(y_test)

# MAE
mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE: ₹{round(mae)}")

Baseline MAE: ₹468748
